# Setting up or sample 
We'll first take the first sample, from the first class type 0. 

In [ ]:
import numpy as np
import matplotlib as plt
data = np.load("../CodeTest/fashion_mnist_train.npz")
X, y = data["X"], data["y"]

data = np.load("../CodeTest/fashion_mnist_test.npz")
X_test, y_test = data["X"], data["y"]

np.set_printoptions(linewidth=120, threshold=np.inf, suppress=True)
sample = X[0]
subset = sample[:28, :28]
print(subset)

[[  0   0   0   0   0   1   0   0   0   0  41 188 103  54  48  43  87 168 133  16   0   0   0   0   0   0   0   0]
 [  0   0   0   1   0   0   0  49 136 219 216 228 236 255 255 255 255 217 215 254 231 160  45   0   0   0   0   0]
 [  0   0   0   0   0  14 176 222 224 212 203 198 196 200 215 204 202 201 201 201 209 218 224 164   0   0   0   0]
 [  0   0   0   0   0 188 219 200 198 202 198 199 199 201 196 198 198 200 200 200 200 201 200 225  41   0   0   0]
 [  0   0   0   0  51 219 199 203 203 212 238 248 250 245 249 246 247 252 248 235 207 203 203 222 140   0   0   0]
 [  0   0   0   0 116 226 206 204 207 204 101  75  47  73  48  50  45  51  63 113 222 202 206 220 224   0   0   0]
 [  0   0   0   0 200 222 209 203 215 200   0  70  98   0 103  59  68  71  49   0 219 206 214 210 250  38   0   0]
 [  0   0   0   0 247 218 212 210 215 214   0 254 243 139 255 174 251 255 205   0 215 217 214 208 220  95   0   0]
 [  0   0   0  45 226 214 214 215 224 205   0  42  35  60  16  17  12  13  70   

First, we must understand what we need to pass in whenever we have a convolutional block. In general the block consists of a the input image, a filter being slid onto the image where we iterate stride by stride, and an output feature map that is the result of the convolution operation at each given stride. Then, we apply an activation function, and finally we apply pooling to reduce the spatial dimensions for further layers. 
For now, we'll cover the convolutional block, and the forward pass before trying to tackle the backward pass of the model. 
Since for *my* project, I would like to use this model with color, input_shape is used for rgb values.

In [ ]:
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = np.zeros((num_filters, 1)) 

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = np.sqrt(2.0 / n)
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = np.random.randn(
            filter_size[0],         #width
            filter_size[1],         #height
            input_depth,            #depth 
            num_filters             #number of filters
        )* std
    

Inside of our __init__ method, we have passed in a few things where:
* self: keyword to allow the object to save local copys of an data using the "self." prefix.
* input_shape: A list of 4 elements, (samples, width, height, depth). Where width and height are the width and height of the input image, and depth being either 1 or 3, depending on if we're using greyscale or RGB, respectively. 
* num_filters: The number of unique filters we plan on passing through our model. In general, each unique filter should ideally learn to detect a different feature type. 
  * One filter might detect vertical edges.
  * One filter might detect horizontal edges.
  * One filter might detect diagonal edges.
  * Another filter may pick up on textures, corners, or blobs. 
* filter_size: A list for how large our *window* will be each time we slide it through our input image. Wtih a large filter size e.g (11, 11) we'd consider using *Fast Fourier Transform Convolution* to speedup our training time, since the overhead of using said algorithm is countered when our filter size is large. However, for smaller ones, I'll use pointwise multiplication (standard convolution operation). 
* strides: A list for how much the filter should move across each iteration horizontally and vertically. 
* padding: A string used to define how we'd like to set the padding of the image. Of course, using "same" padding will net us a border of a set size around the image such that we don't lose any edge detail. 

Next, we need to set the bias for each filter by doing:
```python
self.biases = np.zeros((num_filters, 1))
```
where we create a 2d array of zeros where each row corresponds to a unique filter, and since there is only ever one bias we denote that by using 1 as our column. 

Now, we need to handle the logic of initalizing the values inside of each filter. Since I used **He Inialization** for my dense layers, I will use the same initialziation here. We'll use the formula 
$$
\large W \rightarrow \mathcal{N} (0, \sqrt{\frac{2}{n}})
$$
where:
* $\mathcal{N} \rightarrow $ is the Gaussian probability distribution. 
* $0 \rightarrow $ is the mean of the distribution
* $\sqrt{\frac{2}{n}} \rightarrow$ is the standard deviation
* $n \rightarrow$ is the number of inputs to the node. 

You could think of n as the equation:
$$n = F \times F \times D 
$$
where:
* $F \rightarrow $ filter height and depth
* $D \rightarrow $ the number of input channels. 

```python
n = self.filter_size[0] * self.filter_size[1] * input_depth
std = np.sqrt(2.0 / n)
self.filter_weights = np.random.randn(filter_size[0],
                                       filter_size[1], 
                                              input_depth, num_filters) * std


# Forward Pass
Inside our forward pass we'll need to do the following things: 
1. Extract the input dimensions ($S, W_{in}, H_{in}, D_{in}$)  
  Note: S equals the number of samples passed in. It could either be a batch of samples given from Model.train() or the entire dataset.
2. Calculate the output dimensions using the stride, padding, and fiter size: 
$$
W_{out} = \frac{W_{in} + 2P-F_W}{S_W} + 1
$$
$$
H_{out} = \frac{H_{in} + 2P-F_H}{S_H} + 1
$$
$$
D_{out} = numfilters
$$
 Notice how the output depth (number of channels in the output feature map), is equal to the number of filters the provided.
 Why?
 Each filter produces one channel in the output:
 Ex:
  * Input: $ 32 \times 32 \times 3 \text{RGB} $
    * So H_in = 32, W_in = 32, and D_in = 3
  * Filters: $ 8 \text{filters, each } 3 \times 3\ times 3 $
  * Output: $ 30 \times 30 \times 8 $ (assuming valid padding and stride = 1)



3. Pad input image (our borders of zeros per say)
4. Create an output tensor (our output feature map) with zeros using shape($S, H_{out}, W_{out}, D_{out}$), initialized to zeros. 
  * To get D_out, image passing in two filters into the same sample image, we must first perform the whole convolution pass over the image using filter_1, then perform the same pass now using the second filter. In the end, we'd be storing 2 output feature maps, one with the first filter, and another with the second filter. 
5. Iterate over the spatial positions using the stride
6. In each position:
 * Slice the input window (create a small submatrix).
 * Perform elementwise multiplication between the filter and window.
 * sum the result plus bias
 * return this scaler to the output tensor.
7. Save the input image for backpropogation
In this example, the init method as well as steps 1-4 are completed. However, since steps 5 and 6 are quite complex, this will be implemented later in the notebook 

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        #input_shape has form (batch_size, height, width, channels)
        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = np.zeros((num_filters, 1)) 

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = np.sqrt(2.0 / n)
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = np.random.randn(
            filter_size[0],         #width
            filter_size[1],         #height
            input_depth,            #depth 
            num_filters             #number of filters
        )* std

    def forward(self, inputs):
        #Extract Input dimensions
        S, H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            P = (self.filter_size[0] - 1) // 2
        else:            
            P = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * P - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * P - self.filter_size[1]) / self.strides[1] + 1)
        
        #(0, 0) -> don't touch the number of samples in the batch
        #(P, P) -> pad top and bottom pixels by P pixels (axis 1)
        #(P, P) -> pad left and right pixels by P pixels (axis 2)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = np.pad(array = inputs, 
                            pad_width = ((0, 0), (P, P), (P, P), (0, 0)),
                            mode = 'constant',
                            constant_values = 0)

        #Create an output tensor of size (batch_size, H_out, W_out, D_out)
        self.output_tensor = np.zeros(self.input_shape[0], H_out, W_out, self.num_filters)

        #create our sliding window
        #create the logic for convolution operation
        #save the output tensor using self. for backpropogation
        
        pass

# Np.einsum
In a neural network, it is never ideal to work on **only** one sample at a time. If we were to perform a naive implementation of a CNN architecture, we'd have to create 6 nested for loops with complexity 

$$
O(S \times F \times H_{out} \times W_{out} \times D_{in} \times {F_H} \times {F_W})
$$
where:  
$S$: Number of samples passed into the pass  
$F$: number of filters being passed  
$H_{out}$: height of the output tensor  
$W_{out}$: width of the output tensor  
$D_{in}$: Number of channels we must work through  
$F_{H}$: height of the feature  
$F_{W}$: Width of the feature.  

Quite frankly, performing all of this computation inside Python loops is extremely inefficient, because Python cannot optimize nested numerical loops. In the end, we'd end up sequentially looping through samples and increase overhead. 

Even though the theoretical complexity 
$𝑂(⋅)$ remains the same after vectorization, we can make the process orders of magnitude faster by leveraging NumPy’s backend (written in C and Fortran) to perform the same operations in parallel and with optimized memory access.

To achieve this, we use np.einsum, which lets us express complex summations and tensor contractions in Einstein notation.
This effectively pushes the heavy lifting of the convolution operation from Python into highly optimized, vectorized routines — allowing all samples and filters to be processed in parallel.

An issue with the np.einsum is that we have no way to incorporate the sliding window aspect that is needed to actually build the output feature map. 
In this case, we'll use the as_strided method.

# What as_strided does
as_strided lets the user create a **view** of the array with a custom shape and strides. It will give us a submatrix that won't create a deep copy of the original array.

```python

patches = as_strided(
    padded_inputs,
    shape=(H_out, W_out, fH, fW),
    strides=padded_inputs.strides*2)

```
Here, padded_inputs is a list containing the shape of the padded image, where in our shape, we denote how we will move through the input image. 
(H_out, W_out) -> number of output positions along height and width
(fH, fW) -> the size of each patch

In the end, patches[i, j] gives the fH x fW submatrix of the padded input image. 

If we had perform 
patches([0, 0])
$$ \text{if padded image} = 
\begin{bmatrix}
 0 & 0 & 0 & 0 & 0 & 0 \\
 0 & 1 & 2 & 3 & 0 & 0 \\
 0 & 0 & 1 & 2 & 3 & 0 \\
 0 & 3 & 0 & 1 & 2 & 0 \\
 0 & 2 & 3 & 0 & 1 & 0 \\
 0 & 0 & 0 & 0 & 0 & 0
 \end{bmatrix}
$$

$$ \text{Then, padded image[0, 0]} = 
\begin{bmatrix}
0 & 0 & 0 \\
0 & 1 & 2 \\
0 & 0 & 1 
\end{bmatrix}
\quad
\text{padded image[0, 1]} = 
\begin{bmatrix}
0 & 0 & 0 \\
1 & 2 & 3 \\
0 & 1 & 2
\end{bmatrix}
\quad
\text{padded image[1, 0]} = 
\begin{bmatrix}
0 & 1 & 2 \\
0 & 0 & 1 \\
0 & 0 & 1
\end{bmatrix}
$$

Now that we have all the patches, we actually won't pass in the padded input image, instead, we'll pass in the series of patches of shape
(H_out, W_out, fH, fW) with the kenrel having shape (fH, fW). 

Luckily np.einsum follows a simple rule: 
* Indicies that appear in both the left-hand side and the right-hand side are summed over. 
* Indicies that appear only on the left-hand side are kept as output dimensions. 
So, 
```python
output = np.einsum('ijmn, mn -> ij', patches, kernel)
```


| Index | Meaning | 
|----------------------|-----------| 
| i,j |output position (top-left corner of the patch)  |
| m, n | within-patch indicies, to multiply with kernel. |
| -> ij | Keep THESE dimensions, and sum over m, n, producing scaler per output position. |

In english, if patches passes in the indicies of the output feature map with ij, then passes the actual size of the sliding windows of size mn, we would then multiply this by the same dimensions of the kernel, which are also mn.  
In the example below we have a 4x4 input image, where we pad the image to have shape 6x6. After creating the padded image, we'll use the as_strided method to create a sliding window of shape $(H_{out}, W_{out}, fH, fW) $. 

The first two indicies will tell us where our **window** is, and the third and fourth indicies are the resulting size of the array. 

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
# Small 4x4 input image
image = np.array([
    [1, 2, 3, 0],
    [0, 1, 2, 3],
    [3, 0, 1, 2],
    [2, 3, 0, 1]
])

# 3x3 filter
kernel = np.array([
    [ 1,  0, -1],
    [ 1,  0, -1],
    [ 1,  0, -1]
])

H_in, W_in = image.shape
fH, fW = kernel.shape

P = (fH - 1) // 2
padded_inputs = np.pad(array = image, 
                            pad_width = ((P, P), (P, P)),
                            mode = 'constant',
                            constant_values = 0)
#print(padded_inputs)
#looks like
#[[0 0 0 0 0 0]
# [0 1 2 3 0 0]
# [0 0 1 2 3 0]
# [0 3 0 1 2 0]
# [0 2 3 0 1 0]
# [0 0 0 0 0 0]]

H_out = padded_inputs.shape[0] - fH + 1         #4
W_out = padded_inputs.shape[1] - fW + 1         #4

#every (i, j) index of patches corresponds to a sliding window of the padded_inputs
patches = as_strided(
    padded_inputs,
    shape = (H_out, W_out, fH, fW),         #The reulting output should be H_out x W_out, where we slide over it fH, fW at a time. 
    strides = padded_inputs.strides*2
)
print(patches.shape)
#In the output, notice how we print the first sliding window, the second sliding window, then the second row for a sliding window. 
print(patches[1, 0])
#looks like
#[[0 1 2
#  0 0 1
#  0 3 0]]
print(patches[1]) #first row of slides.


output = np.einsum("ijmn, mn -> ij", patches, kernel)

print(output)

(4, 4, 3, 3)
[[0 1 2]
 [0 0 1]
 [0 3 0]]
[[[0 1 2]
  [0 0 1]
  [0 3 0]]

 [[1 2 3]
  [0 1 2]
  [3 0 1]]

 [[2 3 0]
  [1 2 3]
  [0 1 2]]

 [[3 0 0]
  [2 3 0]
  [1 2 0]]]
[[-3 -4  0  5]
 [-3 -2 -2  6]
 [-4  2 -2  3]
 [-3  4  0  1]]


Ex:2
In this example, we'll have two tensors, the first has 3 batches of 2x5 arrays, while the second tensor will have 3 batches of 5x3 arrays. Here, our goal is to perform matrix multiplication where each batch is multiplied by the other batch.  
  
In np.einsum(), we'll first have to denote the input dimensions of a, so we do **ijk**, 
where:  
* i = batch size for a
* j = number of rows in a[i] (j)
* k = dimension we will sum over

In **ikl** we'll again have this order 
where:  
* i = batch size for b (both are equal)
* k = dimension we will sum over
* l = column index in b 

For the output "ijl" we'll have the order where:
* i = the number of batches made after the operation
* j = the number of rows from the operation in each batch
* l = the number of columns from the operation in each batch.

Finally, after denoting indicies we'll pass in a, b

In [9]:
import numpy as np
a = np.random.randn(3, 2, 5) 
b = np.random.randn(3, 5, 3)
c = np.einsum("ijk, ikl -> ijl", a, b)
print(c.shape)  # (3, 2, 3)


(3, 2, 3)


# Implementation of einsum in the forward pass. 

Its important to reiterate what our input shape as well as the shape of our filters.
First, input_shape passes in the data of the model, with the shape of:  
$$
\text{Input shape} = (S, H_{\text{in}}, W_{\text{in}}, D_{\text{in}})
$$
and for the filters we have shape:
$$
\text{Filter Shape} = (fH, fW, D_{out})
$$
where:
* $fH \rightarrow $ is the height of the filter
* $fW \rightarrow $ is the width of the filter
* $D_{out} \rightarrow $ is the number of filters we'll have, taken from num_filters

Using the formulas found in CNN_Idea, we'll get the output feature height, and output feature width. 
If given we had strides
$$
\text{inputs.strides} = S_{stride}, H_{stride}, W_{stride}, D_{stride}
$$

Then our patches shape would be 
$$
\text{patches.shape} = (S, H_{out}, W_{out}, fH, fW, Din)
$$
In code this will look like

```python
patches = as_strided(
    inputs,
    shape=(S, H_out, W_out, fH, fW, D_in),
    strides=(
        inputs.strides[0],  # step between samples
        inputs.strides[1],  # step down a row
        inputs.strides[2],  # step across a column
        inputs.strides[1],  # move down 1 row inside patch
        inputs.strides[2],  # move right 1 col inside patch
        inputs.strides[3],  # step across channels
    )
)
```

Here, we want to keep the sample axis, the row axis, the column axis, and sum over the patch column and rows as well as the channels. 

num_filters = 1, filter_size = (3, 3)

```python
output = np.einsum('shwxyc,xycd->shwd', patches, filters)
```
Now, we'll want to keep shwd, where:
s : number of samples
h : H out height
w : W out width
d : d_out depth. 

our full implemementation will be below

In [5]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        #input_shape has form (batch_size, height, width, channels)
        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = np.zeros(self.num_filters) 

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = np.sqrt(2.0 / n)
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = np.random.randn(
            filter_size[0],         #height
            filter_size[1],         #width
            input_depth,            #depth 
            num_filters             #number of filters
        )* std

    def forward(self, inputs):
        #Extract Input dimensions


        fH, fW = self.filter_size
        sH, sW = self.strides
        S, H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            P = (self.filter_size[0] - 1) // 2
        else:            
            P = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * P - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * P - self.filter_size[1]) / self.strides[1] + 1)
        
        #(0, 0) -> don't touch the number of samples in the batch
        #(P, P) -> pad top and bottom pixels by P pixels (axis 1)
        #(P, P) -> pad left and right pixels by P pixels (axis 2)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = np.pad(array = inputs, 
                            pad_width = ((0, 0), (P, P), (P, P), (0, 0)),
                            mode = 'constant',
                            constant_values = 0)

        #Create an output tensor of size (batch_size, H_out, W_out, D_out)
        self.output = np.zeros((self.input_shape[0], H_out, W_out, self.num_filters))

        #create our sliding window
        patches = as_strided(
            padded_inputs,
            shape=(S, H_out, W_out, fH, fW, D_in),
            strides=(
                padded_inputs.strides[0],       # step between samples
                padded_inputs.strides[1] * sH,  # step down a row
                padded_inputs.strides[2] * sW,  # step across a column
                padded_inputs.strides[1],       # move down 1 row inside patch
                padded_inputs.strides[2],       # move right 1 col inside patch
                padded_inputs.strides[3],       # step across channels
            )
        )

        #Keep the samples, h_out, w_out, and the number of channels out. But, iterate over the patch(x, y) with channels c, and with the number of filters d
        self.output = np.einsum('shwxyc,xycd->shwd', patches, self.filter_weights)
        self.output += self.biases.reshape((1, 1, 1, self.num_filters)) 

        self.inputs = inputs
        self.padded_inputs = padded_inputs
        self.patches = patches
        return self.output.copy()
        #save the output tensor using self. for backpropogation

In [8]:
import numpy as np
data = np.load("../CodeTest/fashion_mnist_train.npz")
X, y = data["X"], data["y"]

print(X.shape) #60000 samples, by a 28 by 28 grid. 
X = X[..., np.newaxis]
print(X.shape) #60000 samples, by a 28 by 28 grid, with channel = 1 for grayscale. 
sample_batch = X[:128]
print(sample_batch.shape) #128 samples, lets pass this into our convolution layer. 

#sample_batch.shape[1:] = (28, 28, 1)
conv = Conv_Layer(input_shape = sample_batch.shape[1:], num_filters = 2, filter_size = (3, 3), strides = (1, 1), padding = "same") 
output = conv.forward(inputs = sample_batch)
print(output.shape)

(60000, 28, 28)
(60000, 28, 28, 1)
(128, 28, 28, 1)
(128, 28, 28, 2)


In [9]:
#Lets try some other test cases 
X_test = np.ones((1, 5, 5, 1))  # single 5x5 image, all ones
conv = Conv_Layer(input_shape=(5, 5, 1), num_filters=1, filter_size=(3, 3), strides=(1, 1), padding="valid")

conv.filter_weights[:] = 1  # set all weights = 1
conv.biases[:] = 0
out = conv.forward(X_test)
print(out)
print(out.shape)


[[[[9.]
   [9.]
   [9.]]

  [[9.]
   [9.]
   [9.]]

  [[9.]
   [9.]
   [9.]]]]
(1, 3, 3, 1)


In [11]:
X_test = np.ones((1, 5, 5, 3))  # 3 channels (like RGB)
conv = Conv_Layer((5, 5, 3), num_filters=1, filter_size=(3, 3), strides=(1, 1), padding="valid")
conv.filter_weights[:] = 1
conv.biases[:] = 0

out = conv.forward(X_test)
print(out[0, :, :, 0])  # Each patch sum = 9 * 3 = 27

[[27. 27. 27.]
 [27. 27. 27.]
 [27. 27. 27.]]


In [12]:
X_test = np.ones((1, 5, 5, 1))
conv = Conv_Layer((5, 5, 1), num_filters=2, filter_size=(3, 3), strides=(1, 1), padding="valid")
conv.filter_weights[:, :, :, 0] = 1
conv.filter_weights[:, :, :, 1] = 2  # 2nd filter doubled
conv.biases[:] = 0

out = conv.forward(X_test)
print(out[0, :, :, 0])  # should be all 9s
print(out[0, :, :, 1])  # should be all 18s

[[9. 9. 9.]
 [9. 9. 9.]
 [9. 9. 9.]]
[[18. 18. 18.]
 [18. 18. 18.]
 [18. 18. 18.]]
